# Pair Dataset for Cosine Similarity

Builds image pairs with labels:

- **label = 1** - two different shots of the **same** vehicle (taken from the multi-image field of a single record)
- **label = 0** - two shots of **different** vehicles

In [ ]:
%pip install -q pymysql python-dotenv requests pillow tqdm

In [ ]:
import os
import csv
import random
import hashlib
from pathlib import Path

import pymysql
import requests
from dotenv import load_dotenv
from tqdm.auto import tqdm

from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Tunables for parallel download
NUM_WORKERS_DL = 64        # I/O-bound; 32-128 is usually a sweet spot
MAX_RETRIES    = 2


SEED = 42
random.seed(SEED)

## Configuration

In [ ]:
load_dotenv(dotenv_path=Path.cwd() / ".env")

DB_CFG = {
    "host":     os.environ["DB_HOST"],
    "port":     int(os.environ.get("DB_PORT", 3306)),
    "user":     os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
    "database": os.environ["DB_NAME"],
}

N_POSITIVE_PAIRS   = 5000     # same-vehicle pairs (label=1)
N_NEGATIVE_PAIRS   = 5000     # different-vehicle pairs (label=0)
IMAGES_PER_VEHICLE = 2        
TOP_K_IMAGES       = 3       
DOWNLOAD_TIMEOUT   = 10

OUT_DIR     = Path("pairs")
IMG_DIR     = OUT_DIR / "images"
PAIRS_CSV   = OUT_DIR / "pairs.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output dir: {OUT_DIR.resolve()}")

## Fetching candidates

We pull records with **at least two** available images. The query is encapsulated inside a helper so that table/column names are not exposed to the rest of the notebook - only anonymized fields are returned.

In [ ]:
def fetch_candidates(limit: int, min_images: int = 2):
    """
    Returns a list of dicts of the form:
        {"vehicle_id": <int>, "image_urls": [<url>, ...]}
    Schema details are kept inside this function.
    """
    conn = pymysql.connect(
        cursorclass=pymysql.cursors.DictCursor,
        charset="utf8mb4",
        use_unicode=True,
        **DB_CFG,
    )
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT Id AS _id, ImagesSources AS _imgs
                FROM Records
                WHERE SourceType = 'market'
                  AND ImagesSources IS NOT NULL
                  AND ImagesSources != ''
                  AND CHAR_LENGTH(ImagesSources) - CHAR_LENGTH(REPLACE(ImagesSources, ';', '')) >= %s
                ORDER BY Id DESC
                LIMIT %s
                """,
                (min_images - 1, limit * 2),
            )
            rows = cur.fetchall()
    finally:
        conn.close()

    out = []
    for r in rows:
        urls = [u for u in (r["_imgs"] or "").split(";") if u.strip()]
        seen, uniq = set(), []
        for u in urls:
            if u not in seen:
                seen.add(u)
                uniq.append(u)
        if len(uniq) >= min_images:
            out.append({"vehicle_id": int(r["_id"]), "image_urls": uniq})
        if len(out) >= limit:
            break

    random.shuffle(out)
    return out


# We need enough unique vehicles for positive pairs + ~2 vehicles per negative pair (with margin)
need_vehicles = N_POSITIVE_PAIRS + N_NEGATIVE_PAIRS // 2 + 500
candidates = fetch_candidates(limit=need_vehicles, min_images=IMAGES_PER_VEHICLE)
print(f"Vehicles with >= {IMAGES_PER_VEHICLE} images: {len(candidates)}")

## Building pairs

In [ ]:
def top_urls(c):
    return c["image_urls"][:TOP_K_IMAGES]

# Positives: two images of the same vehicle
random.shuffle(candidates)

positive_pool = candidates[:N_POSITIVE_PAIRS]
negative_pool = candidates[N_POSITIVE_PAIRS:]

positive_pairs = []
for c in positive_pool:
    urls = top_urls(c)
    if len(urls) < 2:
        continue  # safety net; shouldn't happen given IMAGES_PER_VEHICLE >= 2
    a, b = random.sample(urls, 2)
    positive_pairs.append({
        "vehicle_id_a": c["vehicle_id"],
        "vehicle_id_b": c["vehicle_id"],
        "url_a": a,
        "url_b": b,
        "label": 1,
    })

#Negatives: images of two different vehicles 
negative_pairs = []
attempts = 0
max_attempts = N_NEGATIVE_PAIRS * 10
while len(negative_pairs) < N_NEGATIVE_PAIRS and attempts < max_attempts:
    attempts += 1
    c1, c2 = random.sample(negative_pool if len(negative_pool) >= 2 else candidates, 2)
    if c1["vehicle_id"] == c2["vehicle_id"]:
        continue
    u1 = random.choice(top_urls(c1))
    u2 = random.choice(top_urls(c2))
    negative_pairs.append({
        "vehicle_id_a": c1["vehicle_id"],
        "vehicle_id_b": c2["vehicle_id"],
        "url_a": u1,
        "url_b": u2,
        "label": 0,
    })

all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)

print(f"Positive: {len(positive_pairs)} | Negative: {len(negative_pairs)} & Total: {len(all_pairs)}")

## Downloading images

URLs may repeat across pairs (the same vehicle appears in both members of a positive pair; negatives may also share URLs across different pairs). We cache files on disk using a stable hash of the URL.

In [ ]:
def url_to_filename(url: str) -> str:
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]
    path = urllib.parse.urlparse(url).path
    _, ext = os.path.splitext(path)
    if not ext or len(ext) > 6:
        ext = ".jpg"
    return f"{h}{ext.lower()}"


def make_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=MAX_RETRIES,
        backoff_factor=0.3,
        status_forcelist=(500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
    )
    adapter = HTTPAdapter(
        pool_connections=NUM_WORKERS_DL,
        pool_maxsize=NUM_WORKERS_DL,
        max_retries=retry,
    )
    s.mount("http://",  adapter)
    s.mount("https://", adapter)
    return s


def download_one(session: requests.Session, url: str) -> tuple[str, str | None]:
    dst = IMG_DIR / url_to_filename(url)
    if dst.exists() and dst.stat().st_size > 0:
        return url, str(dst.relative_to(OUT_DIR))
    try:
        r = session.get(url, timeout=DOWNLOAD_TIMEOUT)
        r.raise_for_status()
        dst.write_bytes(r.content)
        return url, str(dst.relative_to(OUT_DIR))
    except Exception:
        return url, None


unique_urls = list({p["url_a"] for p in all_pairs} | {p["url_b"] for p in all_pairs})
print(f"Unique URLs to download: {len(unique_urls)} | workers: {NUM_WORKERS_DL}")

url_to_path: dict[str, str] = {}
failed: list[str] = []

session = make_session()
try:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS_DL) as ex:
        futures = [ex.submit(download_one, session, u) for u in unique_urls]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="download"):
            url, rel = fut.result()
            if rel is not None:
                url_to_path[url] = rel
            else:
                failed.append(url)
finally:
    session.close()

print(f"OK: {len(url_to_path)} | FAIL: {len(failed)}")

## Writing `pairs.csv`

Columns: `path_a, path_b, label, vehicle_id_a, vehicle_id_b`.
Paths are relative to `OUT_DIR` (`pairs/`).

In [ ]:
rows = []
skipped = 0
for p in all_pairs:
    pa = url_to_path.get(p["url_a"])
    pb = url_to_path.get(p["url_b"])
    if not pa or not pb:
        skipped += 1
        continue
    rows.append({
        "path_a": pa,
        "path_b": pb,
        "label": p["label"],
        "vehicle_id_a": p["vehicle_id_a"],
        "vehicle_id_b": p["vehicle_id_b"],
    })

with open(PAIRS_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["path_a", "path_b", "label", "vehicle_id_a", "vehicle_id_b"])
    w.writeheader()
    w.writerows(rows)

pos = sum(1 for r in rows if r["label"] == 1)
neg = sum(1 for r in rows if r["label"] == 0)
print(f"Pairs written: {len(rows)} (pos={pos}, neg={neg}), skipped: {skipped}")
print(f"CSV: {PAIRS_CSV.resolve()}")

## Train / val / test split

We split **by `vehicle_id`** so that a single vehicle cannot appear in both train and test (otherwise we would leak identity through the embeddings and overestimate cosine accuracy).

In [ ]:
SPLIT_RATIOS = {"train": 0.7, "val": 0.15, "test": 0.15}

all_vehicle_ids = sorted({r["vehicle_id_a"] for r in rows} | {r["vehicle_id_b"] for r in rows})
random.shuffle(all_vehicle_ids)

n = len(all_vehicle_ids)
n_train = int(n * SPLIT_RATIOS["train"])
n_val   = int(n * SPLIT_RATIOS["val"])

train_ids = set(all_vehicle_ids[:n_train])
val_ids   = set(all_vehicle_ids[n_train:n_train + n_val])
test_ids  = set(all_vehicle_ids[n_train + n_val:])

def assign(r):
    a, b = r["vehicle_id_a"], r["vehicle_id_b"]
    if a in train_ids and b in train_ids: return "train"
    if a in val_ids   and b in val_ids:   return "val"
    if a in test_ids  and b in test_ids:  return "test"
    return None  # drop cross-split pairs

splits = {"train": [], "val": [], "test": []}
for r in rows:
    s = assign(r)
    if s is not None:
        splits[s].append(r)

for name, lst in splits.items():
    out_csv = OUT_DIR / f"pairs_{name}.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["path_a", "path_b", "label", "vehicle_id_a", "vehicle_id_b"])
        w.writeheader()
        w.writerows(lst)
    pos = sum(1 for r in lst if r["label"] == 1)
    neg = sum(1 for r in lst if r["label"] == 0)
    print(f"{name}: total={len(lst)} (pos={pos}, neg={neg}) -> {out_csv.name}")

## (Optional) Pack into a zip archive

In [ ]:
# shutil.make_archive("pairs_dataset", "zip", OUT_DIR)
# print("Done: pairs_dataset.zip")